# Statistical Reliability

This notebook checks whether the existing ResNet-18 and EfficientNet-B0 test results support a paired McNemar comparison. It does not train or evaluate either model.

McNemar's test requires one correctness indicator per test image for each model. Aggregate metrics and directed error counts cannot determine the paired discordant counts, so no statistic or p-value is calculated when those individual predictions are absent.

In [4]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else (cwd / 'pixels-to-water' if (cwd / 'pixels-to-water' / 'results').exists() else cwd)
RESULTS_DIR = PROJECT_ROOT / 'results'
ALPHA = 0.05
print(f'Project root: {PROJECT_ROOT}')
print(f'Significance level: alpha = {ALPHA}')

Project root: c:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water
Significance level: alpha = 0.05


In [2]:
test_manifest = pd.read_csv(RESULTS_DIR / 'test.csv')
with open(RESULTS_DIR / 'resnet18_test_metrics.json', encoding='utf-8') as file:
    resnet_metrics = json.load(file)
with open(RESULTS_DIR / 'efficientnet_b0_test_metrics.json', encoding='utf-8') as file:
    efficientnet_metrics = json.load(file)

n_test = len(test_manifest)
resnet_correct = round(resnet_metrics['test_accuracy'] * n_test)
efficientnet_correct = round(efficientnet_metrics['test_accuracy'] * n_test)
summary = pd.DataFrame({
    'Model': ['ResNet-18', 'EfficientNet-B0'],
    'Correct predictions': [resnet_correct, efficientnet_correct],
    'Incorrect predictions': [n_test - resnet_correct, n_test - efficientnet_correct],
})
display(summary)
print(f'Number of test samples: {n_test}')

,Model,Correct predictions,Incorrect predictions
0,ResNet-18,343,157
1,EfficientNet-B0,371,129


Number of test samples: 500


In [3]:
prediction_files = sorted(RESULTS_DIR.glob('*prediction*.csv')) + sorted(RESULTS_DIR.glob('*predictions*.csv'))
print('Candidate individual-prediction files:', [path.name for path in prediction_files] or 'none')

if not prediction_files:
    paired_table = pd.DataFrame(
        [['unavailable', 'unavailable'], ['unavailable', 'unavailable']],
        index=['ResNet Correct', 'ResNet Wrong'],
        columns=['EfficientNet Correct', 'EfficientNet Wrong'],
    )
    display(paired_table)
    statistic = None
    p_value = None
    conclusion = (
        'McNemar\'s test cannot be performed from the saved results. '
        'The individual per-image predictions needed to determine the paired '
        'comparison counts are missing; no statistic or p-value is reported.'
    )
else:
    raise NotImplementedError(
        'A candidate prediction file exists. Inspect and verify its schema before using it; '
        'this notebook does not infer or invent a pairing.'
    )

display(Markdown(f'**McNemar test statistic:** `{statistic}`\n\n**p-value:** `{p_value}`\n\n**Interpretation:** {conclusion}'))

Candidate individual-prediction files: none


,EfficientNet Correct,EfficientNet Wrong
ResNet Correct,unavailable,unavailable
ResNet Wrong,unavailable,unavailable


**McNemar test statistic:** `None`

**p-value:** `None`

**Interpretation:** McNemar's test cannot be performed from the saved results. The individual per-image predictions needed to determine the paired comparison counts are missing; no statistic or p-value is reported.